# B2.14 · Bonus — Google Mantis, the pipeline in production

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B2.13 · Attesting control intent for agents and MCP servers](https://spbreed.github.io/cyber-commons/lessons/B2.13.html)**.

| | |
|---|---|
| Tools used | Google Mantis, OpenGrep, GLM-4.6, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Map Mantis onto the 15 stages, parse its two output shapes, and score a sample against a held-out key.

**Why a security engineer needs it.** A reference implementation is adopted as a product, and its outputs are trusted without an eval. The control it builds is: map Mantis's stages onto the pipeline you built, then score it with your own held-out key before trusting it.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Somebody has already built this pipeline and published what happened. Reading it is worth an afternoon; adopting it without scoring it against a held-out key is how a reference implementation becomes a dependency you cannot evaluate.

> **At CyberTravels.** Somebody else has already built this pipeline and published what happened. Adopting it without scoring it against a held-out key is how a reference implementation becomes a dependency CyberTravels cannot evaluate.

## 2 · The framework

```
   published pipeline            your pipeline
   +------------------+          +------------------+
   | stages 1..15     |  map ->  | stages 1..15     |
   +------------------+          +------------------+
            |
       score it against a HELD-OUT key
            |
   +--------v---------+
   | adopt / adapt /  |
   | leave it alone   |
   +------------------+

   a reference implementation is a starting point you evaluate
```

**Bonus.** You have now built all fifteen stages. This lesson looks at a real
implementation of the same pipeline — **[Google Mantis](https://github.com/google/mantis)**
— and does the one thing that matters before adopting any of them: maps its
stages onto yours, then **scores it with your own held-out key.**

Two things are worth understanding about Mantis specifically.

**It is model-agnostic.** Mantis ships security-review *skills* for coding
agents rather than a bundled model. That is the same architecture as this track:
the pipeline is the product, the model is a component. It means you can run it
on open weights — GLM-4.6, Kimi K2 — which is what makes it usable without a
frontier account.

**It has two output shapes**, and they serve different stages:

- a **`learning_entry`** — appended to a historical learnings file, feeding
  stage 1 (historical parsing) on the next run;
- a **`finding`** object — a vulnerability report, feeding stages 8–10.

That first shape is the interesting one. It closes the loop from Phase 5 back to
Phase 1, which is the property that turns a pipeline into something that
improves.

The bonus framing is deliberate: a reference implementation is a **starting
point you evaluate**, not a product you trust. C2.6 gave you the tools;
this is where you point them at someone else's pipeline.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Map Mantis onto the fifteen stages

Adoption starts with the coverage question: which stages does it do, which does it assume you already have, and which are still yours?

## 4 · Parse the two output shapes

Before scoring anything you have to ingest it. Both shapes are JSON; the `history` field on a learning entry is required and is the one most commonly missing in a first integration.

## 5 · Score it against a held-out key

This is the whole point of the bonus. Conformance is structural — with structured output it goes to 1.00 and says nothing about quality. The number that decides adoption is expert accuracy against a key the tool never saw, matched on **parent directory plus filename**.

## 6 · The stage, as a skill

Google's Mantis is a set of claims: these stages, this output shape, this accuracy. The skill checks all three — maps it onto the stage model, conformance-checks its published samples, and scores its findings against a held-out key.

### The skill — [`skills/appsec/reference-pipeline-scoring/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/reference-pipeline-scoring/SKILL.md)

```yaml
name: reference-pipeline-scoring
description: >-
  Map a published or vendor pipeline onto the stage model, check its output
  against the schema it claims, and score its findings against a held-out key.
  Use when evaluating a reference implementation, a security product's agent, or
  any pipeline you are asked to adopt.
allowed-tools: Read, Grep, Glob
```

# A reference implementation is something you evaluate

Somebody else's pipeline is a set of claims: these stages, this output shape,
this accuracy. All three are checkable, and checking them is cheaper than
adopting and discovering. The interesting result is usually not the score — it
is the phase the pipeline does not cover at all.

## When to use this

Before adopting a published pipeline, when comparing two, and when a vendor
claims a number you are expected to plan around.

## Procedure

**1 — Map it onto the stage model.** For each stage, does the pipeline cover it
strongly, weakly, or not at all? Coverage claims are usually accurate about the
stages the pipeline is proud of and silent about a whole phase.

**2 — Take its output and check conformance to its own schema.** Required
fields present, enumerated values in range, nothing that is prose where an
object was promised. Conformance failures in a published sample are the cheapest
finding available.

**3 — Score against a held-out key.** Not the examples the pipeline ships. Match
findings to truth by location and defect class, then compute precision and
recall, and report both — a pipeline can look excellent on either alone.

**4 — Separate the model's errors from the harness's.** A null CWE is a schema
problem; a finding at the wrong location is an analysis problem. They have
different fixes and different owners.

**5 — Report the uncovered phase as the headline.** A pipeline that scores well
on the stages it implements is still a partial answer, and the gap is what you
would have to build.

## Output contract

```json
{
  "stages": [{"stage": 0, "coverage": "strong|weak|none"}],
  "uncovered_phases": ["str"],
  "conformance": {"samples": 0, "conforming": 0, "failures": [{"sample": 0, "reason": "str"}]},
  "score": {"matched": 0, "precision": 0.0, "recall": 0.0, "key": "held-out"},
  "errors": {"schema": 0, "analysis": 0}
}
```

## Failure modes

- **Scoring on the shipped examples.** They were chosen.
- **Reporting one of precision and recall.** Either alone flatters.
- **Treating uncovered stages as out of scope.** They are the work you inherit.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/reference-pipeline-scoring/scripts/reference_pipeline_scoring.py
SCRIPT = "skills/appsec/reference-pipeline-scoring/scripts/reference_pipeline_scoring.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The stage map shows Mantis covering stage 7 strongly with a stage-1 learning loop, and not covering Phase 4 at all. Three of five sample outputs conform — one learning entry is missing the required `history` field, one finding has a null CWE, and one is prose. Scored against the held-out key, expert accuracy is below 1.0: one correct, one half credit for the null class, and one missed finding Mantis never reported. The learning entry then feeds the next run's risk zones.

## Your turn

Run the real thing: clone `google/mantis`, point it at a repository you have ground truth for, and score its output with a scoring harness. The gap between its conformance and its expert accuracy on *your* code is the only number that should decide whether you adopt it.

## Where this leaves you

**What you can do now.** A harness you can name the eight parts of, evaluate on a corpus with known answers rather than on how confident it sounds, price per confirmed finding across a run nobody watched, and salt with bait that has no false positives.

**What you still cannot do.** Everything you have built so far is defensive and cooperative: it runs against systems that are not trying to defeat it. You have no evidence about how any of it behaves against someone who is — including the evaluation you have been trusting.

**Function C attacks it, starting with the loop pointed the other way round. Next → C1.0, what red teaming and research with AI means.**

---

**Next → [C1.0 · Start here — what red teaming and research with AI means](https://spbreed.github.io/cyber-commons/lessons/C1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.14.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.14.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*